<h1>DOCUMENT LOADER</h1>

In [18]:
from langchain_community.document_loaders import Docx2txtLoader
import os

def document_loader(folder_path):

    document = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith('.docx'):
            file_path = os.path.join(folder_path,file_name)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()

            document.extend(pages)

    return document

<h1>Text Splitter</h1>

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(document):

    textsplitter = RecursiveCharacterTextSplitter(
        chunk_size = 600,
        chunk_overlap = 80
    )

    chunks = textsplitter.split_documents(document)

    return chunks

<H1>VECTOR DATABASE</H1>

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


def create_vector_db(chunk):

    embedding = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

    vector_db = FAISS.from_documents(
        
        chunk,
        embedding
    )

    return vector_db



<h1>CREATE KEYWORD SEARCH</h1>

In [31]:
from rank_bm25 import BM25Okapi

class keyword_search:
    def __init__(self,chunk):
        self.chunk = chunk

        tokenized_chunk = []

        for c in chunk:
            token = c.page_content.lower().split()
            tokenized_chunk.append(token)

        self.bm250 = BM25Okapi(tokenized_chunk)
    def search(self,question,k=5):
        
        query = question.lower().split()
        scores = self.bm250.get_scores(query)

        ranked_answer = sorted(range(len(scores)),
                                key = lambda index:scores[index],
                                reverse=True)

        top_answers = ranked_answer[:k]

        result = []
        for r in top_answers:
                result.append(self.chunk[r])
        return result

<H1>HYBRID RETRIVER</H1>

In [32]:
class HYBRID:

    def __init__(self,vector_db,bm250):
        self.vector_db =vector_db
        self.bm250 = bm250

    def combined_result(self,query,k=5):
        vector_result = self.vector_db.similarity_search(query,k=k)
        bm250_result = self.bm250.search(query,k=k)

        combinedResult = vector_result + bm250_result

        answer = []
        seen_content = set()

        for c in combinedResult:
            if c.page_content not in seen_content:
                answer.append(c)
                seen_content.add(c.page_content)

        return answer

<h1>RERANKER</h1>

In [33]:
from sentence_transformers import CrossEncoder

class ReRanker:
    def __init__(self):
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self,question,document,k=5):

        pairs = []

        for c in document:

            pairs.append([
                question,
                c.page_content

            ])
        scores = self.model.predict(pairs)

        ranked_answers = sorted(zip(scores,document),
                            key = lambda x:x[0],
                            reverse=True)

        top_answers = ranked_answers[:k]

        result = []

        for scores,document in top_answers:
                result.append(document)
        return result


<h1>PROMPT</h1>

In [34]:
from langchain_groq  import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


def generate_answers(query,document,k=5):

    context = ''
    for c in document:
        context += c.page_content
        context += "\n\n"

    prompt = ChatPromptTemplate.from_template(
        '''
        Answer from the context.

        if the question is out of the context. just say,"i dont know"

        context
        {context}

        question
        {question}
        '''
    )
    llm_model = ChatGroq(
        model = 'openai/gpt-oss-20b',
        temperature = 0
    )

    chain = prompt | llm_model

    response = chain.invoke(
        {
            'context': context,
            'question':query
        }
    )

    return response.content
     

<h1>GATHERING THE PIPELINE</h1>

In [35]:
import gradio as gr

#document loader
folder_path = 'E:\GEN-AI-PROJECTS'

document = document_loader(folder_path)

# text splitting 
textSplitter = text_splitter(document)

#create vector database
vectrDatabase = create_vector_db(textSplitter)

# bm250 
bm250 = keyword_search(textSplitter)

# hybrid search

hybridSearch = HYBRID(vectrDatabase,bm250)

#rerank
reranker = ReRanker()

def Generate_Answers(query):

    retirved_answer = hybridSearch.combined_result(query,k=5)

    ranked_answer = reranker.rerank(query,retirved_answer,k=10)


    answer = generate_answers(query,retirved_answer,k=5)

    sources = ""

    for i,document in enumerate(retirved_answer):
        pages = document.metadata.get(
            'pages',
            'unknown'
        )

        sources += f'\nSource {i+1}: Page {pages}'
        final_response = answer
        final_response += '\n\nSources'
        final_response += sources

    return final_response



demo = gr.Interface(
    fn = Generate_Answers,
    inputs=gr.Textbox(
        label = 'enter your question'
    ),
    outputs=gr.Textbox(
        label = 'Answer',
        lines = 15
    ),
    
        title = 'DEEP KNOWLWDGE',
        description='Advanced RAG Knowledge Base'



)

demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
